In [1]:
import numpy as np
import json
from pathlib import Path
from collections import defaultdict
import random

# Get classification label from file path
def extract_label_from_path(video_path):
    path_str = str(video_path).lower()
    
    if path_str.startswith('fall/') or path_str.startswith('fall\\'):
        return 1  # Fall
    elif path_str.startswith('no-fall') or path_str.startswith('no_fall'):
        return 0  # No fall
    else:
        # Try to infer from path
        if 'fall' in path_str and 'no' not in path_str:
            return 1
        elif 'no-fall' in path_str or 'no_fall' in path_str:
            return 0
        else:
            raise ValueError(f"Cannot determine label from path: {video_path}")

# MAIN
clips_metadata_file = "data/segmented_clips/clips_metadata.json"
output_dir = "data"
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

# Split configuration
train_ratio = 0.7 # 70% of videos for training
val_ratio = 0.15 # 15% for validation
test_ratio = 0.15 # 15% for testing
random_seed = 42 # For reproducibility

assert abs(train_ratio + val_ratio + test_ratio) == 1, "Ratios must sum to 1.0"

# Load clips metadata
with open(clips_metadata_file, 'r') as f:
    clips_metadata = json.load(f)

print(f"Loaded {len(clips_metadata)} total clips\n")

# Group clips by original video
video_to_clips = defaultdict(list)
for clip_info in clips_metadata:
    video_name = clip_info['original_video']
    
    # Add label to clip info
    label = extract_label_from_path(clip_info['original_video_category'])
    clip_info['label'] = label
    
    video_to_clips[video_name].append(clip_info)

# Get list of unique videos
video_names = list(video_to_clips.keys())
num_videos = len(video_names)

print(f"Total videos: {num_videos}")
print(f"Total clips: {len(clips_metadata)}")
print(f"Average clips per video: {len(clips_metadata) / num_videos:.1f}\n")

# Count fall vs no-fall
fall_clips = sum(1 for c in clips_metadata if c['label'] == 1)
no_fall_clips = len(clips_metadata) - fall_clips
print(f"Fall clips: {fall_clips} ({fall_clips/len(clips_metadata)*100:.1f}%)")
print(f"No-fall clips: {no_fall_clips} ({no_fall_clips/len(clips_metadata)*100:.1f}%)\n")

# Shuffle videos
random.seed(random_seed)
random.shuffle(video_names)

# Calculate split indices
train_idx = int(num_videos * train_ratio)
val_idx = train_idx + int(num_videos * val_ratio)

# Split videos
train_videos = video_names[:train_idx]
val_videos = video_names[train_idx:val_idx]
test_videos = video_names[val_idx:]

# Collect clips for each split
train_clips = []
val_clips = []
test_clips = []

for video in train_videos:
    train_clips.extend(video_to_clips[video])

for video in val_videos:
    val_clips.extend(video_to_clips[video])

for video in test_videos:
    test_clips.extend(video_to_clips[video])

# Count fall/no-fall per split
def count_labels(clips):
    fall = sum(1 for c in clips if c['label'] == 1)
    no_fall = len(clips) - fall
    return fall, no_fall

train_fall, train_no_fall = count_labels(train_clips)
val_fall, val_no_fall = count_labels(val_clips)
test_fall, test_no_fall = count_labels(test_clips)

# Print split statistics
print("=" * 70)
print("SPLIT STATISTICS:")
print("=" * 70)

print(f"\nTrain Set:")
print(f"  Videos: {len(train_videos)} ({len(train_videos)/num_videos*100:.1f}%)")
print(f"  Clips: {len(train_clips)} ({len(train_clips)/len(clips_metadata)*100:.1f}%)")
print(f"    Fall: {train_fall}, No-fall: {train_no_fall}")

print(f"\nValidation Set:")
print(f"  Videos: {len(val_videos)} ({len(val_videos)/num_videos*100:.1f}%)")
print(f"  Clips: {len(val_clips)} ({len(val_clips)/len(clips_metadata)*100:.1f}%)")
print(f"    Fall: {val_fall}, No-fall: {val_no_fall}")

print(f"\nTest Set:")
print(f"  Videos: {len(test_videos)} ({len(test_videos)/num_videos*100:.1f}%)")
print(f"  Clips: {len(test_clips)} ({len(test_clips)/len(clips_metadata)*100:.1f}%)")
print(f"    Fall: {test_fall}, No-fall: {test_no_fall}")

# Save split information
split_info = {
    'train': {
        'videos': train_videos,
        'clips': train_clips,
        'num_videos': len(train_videos),
        'num_clips': len(train_clips),
        'num_fall': train_fall,
        'num_no_fall': train_no_fall
    },
    'val': {
        'videos': val_videos,
        'clips': val_clips,
        'num_videos': len(val_videos),
        'num_clips': len(val_clips),
        'num_fall': val_fall,
        'num_no_fall': val_no_fall
    },
    'test': {
        'videos': test_videos,
        'clips': test_clips,
        'num_videos': len(test_videos),
        'num_clips': len(test_clips),
        'num_fall': test_fall,
        'num_no_fall': test_no_fall
    },
    'config': {
        'train_ratio': train_ratio,
        'val_ratio': val_ratio,
        'test_ratio': test_ratio,
        'random_seed': random_seed,
        'sequence_length': 60,
        'overlap': 0.0
    }
}

split_file = output_path / 'dataset_split.json'
with open(split_file, 'w') as f:
    json.dump(split_info, f, indent=2)

print(f"\n{'='*70}")
print(f"Split information saved to: {split_file}")
print(f"{'='*70}")

Loaded 7198 total clips

Total videos: 5205
Total clips: 7198
Average clips per video: 1.4

Fall clips: 3211 (44.6%)
No-fall clips: 3987 (55.4%)

SPLIT STATISTICS:

Train Set:
  Videos: 3643 (70.0%)
  Clips: 5019 (69.7%)
    Fall: 2280, No-fall: 2739

Validation Set:
  Videos: 780 (15.0%)
  Clips: 1106 (15.4%)
    Fall: 474, No-fall: 632

Test Set:
  Videos: 782 (15.0%)
  Clips: 1073 (14.9%)
    Fall: 457, No-fall: 616

Split information saved to: data/dataset_split.json
